# KPI Analysis & Reporting

### This notebook analyzes standardized CMS hospital readmissions metrics to identify national trends, state-level performance differences, reporting limitations, and hospital benchmarking opportunities.

## Project Overview

This notebook analyzes CMS hospital readmissions data to evaluate national healthcare readmission trends, state-level benchmarking differences, and hospital-level KPI variation across major readmission measures.

The project was developed using Databricks SQL and notebook workflows to prepare healthcare KPI datasets for downstream Power BI reporting and visualization.

The analysis focuses on:

- Excess Readmission Ratio (ERR)
- Predicted vs Expected Readmission Rates
- Reporting quality and suppression logic
- Facility-level benchmarking variation
- Healthcare KPI reporting preparation

## KPI Overview

In [0]:
%sql
SELECT
    COUNT(DISTINCT facility_id) AS hospitals,
    COUNT(*) AS reporting_records,
    COUNT(DISTINCT measure_name) AS measures
FROM workspace.default.cms_readmissions_clean4;

hospitals,reporting_records,measures
3055,18330,6


In [0]:
%sql
SELECT DISTINCT measure_display_name
FROM workspace.default.cms_readmissions_clean4
ORDER BY measure_display_name;

measure_display_name
COPD
Coronary Artery Bypass Graft Surgery (CABG)
Heart Attack
Heart Failure
Hip/Knee Replacement
Pneumonia


## National KPI Analysis


In [0]:
%sql

SELECT measure_display_name, 
    round(avg(excess_readmission_ratio_numeric),3) AS avg_err,
    round(avg(Predicted_Readmission_Rate_numeric),2) AS avg_predicted,
    round(avg(Expected_Readmission_Rate_numeric),2) AS avg_expected
FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL
GROUP BY measure_display_name
ORDER BY avg_err DESC


measure_display_name,avg_err,avg_predicted,avg_expected
Hip/Knee Replacement,1.004,5.49,5.46
Heart Attack,1.002,13.42,13.38
Coronary Artery Bypass Graft Surgery (CABG),1.002,10.68,10.65
Pneumonia,1.001,15.6,15.56
COPD,1.001,18.0,17.98
Heart Failure,1.001,19.38,19.35


### National KPI Findings

National hospital readmission performance clustered closely around CMS expected benchmarks, with overall Excess Readmission Ratio (ERR) averages remaining near 1.0 across all analyzed measures. Predicted readmission rates were generally slightly above expected rates, contributing to marginally elevated ERR values nationally. While differences between measures were relatively small, Hip/Knee Replacement demonstrated the highest average ERR among the analyzed conditions.

## State-Level KPI Analysis

In [0]:
%sql
SELECT
    state,

    COUNT(DISTINCT facility_id) AS hospital_count,

    COUNT(*) AS reporting_records,

    ROUND(AVG(excess_readmission_ratio_numeric), 3) AS avg_err,

    ROUND(AVG(Predicted_Readmission_Rate_numeric), 2) AS avg_predicted,

    ROUND(AVG(Expected_Readmission_Rate_numeric), 2) AS avg_expected

FROM workspace.default.cms_readmissions_clean4

WHERE excess_readmission_ratio_numeric IS NOT NULL

GROUP BY state

HAVING COUNT(DISTINCT facility_id) >= 10

ORDER BY avg_err DESC;

state,hospital_count,reporting_records,avg_err,avg_predicted,avg_expected
MA,52,240,1.034,14.91,14.38
NJ,61,272,1.028,16.04,15.62
FL,164,771,1.023,15.78,15.47
IL,109,491,1.019,15.78,15.53
MS,50,185,1.014,15.19,14.95
AL,67,247,1.014,14.66,14.51
WV,22,95,1.012,15.95,15.7
GA,88,346,1.011,15.46,15.31
NV,19,94,1.011,14.8,14.59
CA,253,1038,1.01,15.54,15.32


### Top Performing States

In [0]:
%sql
SELECT
    state,
    ROUND(AVG(excess_readmission_ratio_numeric), 3) AS avg_err
FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL
GROUP BY state
HAVING COUNT(DISTINCT facility_id) >= 10
ORDER BY avg_err ASC
LIMIT 10;

state,avg_err
ID,0.943
ME,0.952
MT,0.954
UT,0.956
OR,0.96
WA,0.964
SD,0.965
KS,0.966
CO,0.967
IA,0.972


### Lowest Performing States

In [0]:
%sql
SELECT
    state,
    ROUND(AVG(excess_readmission_ratio_numeric), 3) AS avg_err
FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL
GROUP BY state
HAVING COUNT(DISTINCT facility_id) >= 10
ORDER BY avg_err DESC
LIMIT 10;

state,avg_err
MA,1.034
NJ,1.028
FL,1.023
IL,1.019
MS,1.014
AL,1.014
WV,1.012
NV,1.011
GA,1.011
CA,1.01


### State-Level Benchmarking Findings

State-level benchmarking analysis showed relatively narrow variation in average ERR values across reporting regions, suggesting that national readmission performance is generally standardized. However, measurable differences still existed between states, indicating potential variation in operational workflows, discharge coordination practices, or reporting populations. States with larger reporting volumes may provide more stable benchmarking comparisons due to broader hospital representation.


## Hospital-Level KPI Analysis

### Benchmarking Methodology

Hospital-level benchmarking analysis excluded:
- suppressed/unavailable reporting records
- records below 25 discharges
- NULL ERR values

These filters were applied to improve reporting consistency and reduce low-volume volatility in KPI comparisons.``

In [0]:

%sql
--Reporting status min/max discharge comparison.
SELECT 
    round(avg(number_of_discharges_numeric),2) AS avg_discharges,
    MIN(Number_of_Discharges_numeric) AS min_discharges,
    MAX(Number_of_Discharges_numeric) AS max_discharges
FROM workspace.default.cms_readmissions_clean4
WHERE Number_of_Discharges_numeric IS NOT NULL

avg_discharges,min_discharges,max_discharges
283.42,0,3672


### Exploratory Validation

In [0]:
%sql
SELECT 
    CASE 
        WHEN number_of_discharges_numeric < 25 THEN 'Under 25'
        WHEN number_of_discharges_numeric <100 THEN '25-99' 
        WHEN number_of_discharges_numeric <500 THEN '100-499'
        ELSE '500+'
    END AS discharge_volumne_group,
    COUNT(*) AS records
FROM workspace.default.cms_readmissions_clean4
WHERE Number_of_Discharges_numeric IS NOT NULL
GROUP BY discharge_volumne_group
ORDER BY records DESC;
    


discharge_volumne_group,records
100-499,5530
25-99,1327
500+,1180
Under 25,205


In [0]:
%sql
SELECT 
  reporting_status,
  count(*) AS records,
  Min(number_of_discharges_numeric) AS min_discharges,
  Max(number_of_discharges_numeric) AS max_discharges
FROM workspace.default.cms_readmissions_clean4
WHERE number_of_discharges_numeric IS NOT NULL
GROUP BY reporting_status


reporting_status,records,min_discharges,max_discharges
Standard Reporting,7798,27,3672
Partial Reporting Period,239,32,1211
Unavailable/Suppressed,205,0,0


In [0]:
%sql
SELECT
    facility_name,
    measure_display_name,
    number_of_discharges_numeric,
    footnote,
    reporting_status
FROM workspace.default.cms_readmissions_clean4
WHERE number_of_discharges_numeric = 0
LIMIT 10;

facility_name,measure_display_name,number_of_discharges_numeric,footnote,reporting_status
COMMUNITY HOSPITAL INC,Heart Attack,0,7,Unavailable/Suppressed
FAYETTE MEDICAL CENTER,Heart Attack,0,7,Unavailable/Suppressed
GREENE COUNTY HOSPITAL,Pneumonia,0,7,Unavailable/Suppressed
GREENE COUNTY HOSPITAL,Heart Failure,0,7,Unavailable/Suppressed
LAKE MARTIN COMMUNITY HOSPITAL,Heart Attack,0,7,Unavailable/Suppressed
BULLOCK COUNTY HOSPITAL,Pneumonia,0,7,Unavailable/Suppressed
TROY REGIONAL MEDICAL CENTER,Hip/Knee Replacement,0,7,Unavailable/Suppressed
HILL HOSPITAL OF SUMTER COUNTY,COPD,0,7,Unavailable/Suppressed
ST JOSEPH'S HOSPITAL,Coronary Artery Bypass Graft Surgery (CABG),0,7,Unavailable/Suppressed
ABRAZO CENTRAL CAMPUS,Hip/Knee Replacement,0,7,Unavailable/Suppressed


Discharge volume analysis showed that records below 25 discharges aligned closely with CMS unavailable/suppressed reporting categories. As a result, hospital benchmarking queries excluded records below this threshold to improve reporting consistency and reduce low-volume volatility.

## Facility-Level Benchmarking Analysis

In [0]:
%sql
SELECT
    facility_name,
    state,
    measure_display_name,
    Number_of_Discharges_numeric,
    round(excess_readmission_ratio_numeric,3) AS err
FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL
AND Number_of_Discharges_numeric >25
ORDER BY err DESC
LIMIT 10

facility_name,state,measure_display_name,Number_of_Discharges_numeric,err
CARLE BROMENN MEDICAL CENTER,IL,Hip/Knee Replacement,292,1.583
HOLY FAMILY HOSPITAL,MA,Hip/Knee Replacement,131,1.566
BROWN UNIVERSITY HEALTH MORTON HOSPITAL,MA,Hip/Knee Replacement,239,1.559
SAINT FRANCIS HOSPITAL MUSKOGEE,OK,Hip/Knee Replacement,102,1.522
HCA FLORIDA OAK HILL HOSPITAL,FL,Hip/Knee Replacement,138,1.52
ST MARY'S HOSPITAL,GA,Hip/Knee Replacement,111,1.51
GOOD SAMARITAN MEDICAL CENTER,FL,Hip/Knee Replacement,101,1.491
ECU HEALTH MEDICAL CENTER,NC,Hip/Knee Replacement,171,1.485
UPMC ST MARGARET,PA,Hip/Knee Replacement,185,1.482
HURON VALLEY-SINAI HOSPITAL,MI,Hip/Knee Replacement,86,1.48


### Facility-Level Benchmarking Findings

Hospital-level benchmarking analysis identified meaningful variation in ERR performance across facilities. To improve reporting consistency and reduce low-volume volatility, suppressed records and hospitals with fewer than 25 discharges were excluded from benchmarking analysis. The resulting dataset demonstrated that facility-level readmission variation persisted even after applying reporting quality filters.

In [0]:
%sql
SELECT
    measure_display_name,

    ROUND(AVG(excess_readmission_ratio_numeric),3) AS avg_err,

    ROUND(STDDEV(excess_readmission_ratio_numeric),3) AS err_variation,

    ROUND(MIN(excess_readmission_ratio_numeric),3) AS min_err,

    ROUND(MAX(excess_readmission_ratio_numeric),3) AS max_err

FROM workspace.default.cms_readmissions_clean4

WHERE excess_readmission_ratio_numeric IS NOT NULL
    AND Number_of_Discharges_numeric >= 25
    AND reporting_status = 'Standard Reporting'

GROUP BY measure_display_name

ORDER BY avg_err DESC;

measure_display_name,avg_err,err_variation,min_err,max_err
Hip/Knee Replacement,1.036,0.223,0.47,1.583
Coronary Artery Bypass Graft Surgery (CABG),1.029,0.114,0.743,1.424
COPD,1.011,0.05,0.868,1.255
Heart Attack,1.01,0.073,0.684,1.264
Pneumonia,1.007,0.069,0.797,1.426
Heart Failure,1.004,0.068,0.714,1.361


### Measure-Level Variation Findings

Measure-level analysis demonstrated that Hip/Knee Replacement had both the highest average ERR and the greatest hospital-level variation among analyzed measures. In contrast, COPD and Heart Failure showed comparatively tighter clustering around benchmark expectations. These findings suggest that some readmission measures may be more operationally sensitive to differences in discharge planning, rehabilitation coordination, and post-surgical recovery workflows.


## ERR Distribution Comparison Across Measures

The following queries compare hospital-level ERR band distributions across multiple measures to evaluate operational consistency and facility-level variation.


These facilities represent the upper end of observed ERR variation for this measure.



In [0]:
%sql
SELECT
    facility_name,
    state,

    Number_of_Discharges_numeric,

    ROUND(excess_readmission_ratio_numeric,3) AS err

FROM workspace.default.cms_readmissions_clean4

WHERE measure_display_name = 'Hip/Knee Replacement'
    AND excess_readmission_ratio_numeric IS NOT NULL
    AND Number_of_Discharges_numeric >= 25
    AND reporting_status = 'Standard Reporting'

ORDER BY err DESC

LIMIT 20;

facility_name,state,Number_of_Discharges_numeric,err
CARLE BROMENN MEDICAL CENTER,IL,292,1.583
HOLY FAMILY HOSPITAL,MA,131,1.566
BROWN UNIVERSITY HEALTH MORTON HOSPITAL,MA,239,1.559
SAINT FRANCIS HOSPITAL MUSKOGEE,OK,102,1.522
HCA FLORIDA OAK HILL HOSPITAL,FL,138,1.52
ST MARY'S HOSPITAL,GA,111,1.51
GOOD SAMARITAN MEDICAL CENTER,FL,101,1.491
ECU HEALTH MEDICAL CENTER,NC,171,1.485
UPMC ST MARGARET,PA,185,1.482
HURON VALLEY-SINAI HOSPITAL,MI,86,1.48


Hospital-level Hip/Knee Replacement ERR values demonstrated substantial variability relative to CMS expected benchmarks, suggesting meaningful differences in post-surgical readmission performance across facilities.

In [0]:
%sql
SELECT
--avg err for knee/hip replacement is 1.036.
    CASE
        WHEN excess_readmission_ratio_numeric >= 1.20 THEN '1.20+'
        WHEN excess_readmission_ratio_numeric >= 1.10 THEN '1.10-1.19'
        WHEN excess_readmission_ratio_numeric >= 1.00 THEN '1.00-1.09'
        ELSE 'Below 1.00'
    END AS err_band,

    COUNT(*) AS hospitals

FROM workspace.default.cms_readmissions_clean4

WHERE measure_display_name = 'Hip/Knee Replacement'
    AND excess_readmission_ratio_numeric IS NOT NULL
    AND Number_of_Discharges_numeric >= 25
    AND reporting_status = 'Standard Reporting'

GROUP BY err_band

ORDER BY err_band DESC;

err_band,hospitals
Below 1.00,114
1.20+,57
1.10-1.19,32
1.00-1.09,43


### ERR Distribution Findings

ERR band analysis showed that COPD readmission performance remained tightly clustered near CMS benchmark expectations across hospitals nationally. Hip/Knee Replacement demonstrated a substantially wider distribution of hospital-level ERR values, including a larger concentration of facilities within elevated ERR bands. This suggests greater operational variability across hospitals for post-surgical readmission management compared to chronic-condition readmission programs.

In [0]:
%sql
SELECT
--avg err for Coronary Artery Bypass Graft Surgery (CABG) 1.029.
    CASE
        WHEN excess_readmission_ratio_numeric >= 1.20 THEN '1.20+'
        WHEN excess_readmission_ratio_numeric >= 1.10 THEN '1.10-1.19'
        WHEN excess_readmission_ratio_numeric >= 1.00 THEN '1.00-1.09'
        ELSE 'Below 1.00'
    END AS err_band,

    COUNT(*) AS hospitals

FROM workspace.default.cms_readmissions_clean4

WHERE measure_display_name = 'Coronary Artery Bypass Graft Surgery (CABG)'
    AND excess_readmission_ratio_numeric IS NOT NULL
    AND Number_of_Discharges_numeric >= 25
    AND reporting_status = 'Standard Reporting'

GROUP BY err_band

ORDER BY err_band DESC;

err_band,hospitals
Below 1.00,140
1.20+,27
1.10-1.19,64
1.00-1.09,124


In [0]:
%sql
SELECT
    CASE
        WHEN excess_readmission_ratio_numeric >= 1.20 THEN '1.20+'
        WHEN excess_readmission_ratio_numeric >= 1.10 THEN '1.10-1.19'
        WHEN excess_readmission_ratio_numeric >= 1.00 THEN '1.00-1.09'
        ELSE 'Below 1.00'
    END AS err_band,

    COUNT(*) AS hospitals

FROM workspace.default.cms_readmissions_clean4

WHERE measure_display_name = 'COPD'
    AND excess_readmission_ratio_numeric IS NOT NULL
    AND Number_of_Discharges_numeric >= 25
    AND reporting_status = 'Standard Reporting'

GROUP BY err_band

ORDER BY err_band DESC;

err_band,hospitals
Below 1.00,651
1.20+,1
1.10-1.19,64
1.00-1.09,790


##Reporting Views

### Reporting Layer Preparation

The following reporting views were created to support downstream Power BI dashboard development. These views summarize KPI metrics into presentation-ready datasets optimized for visualization and executive reporting.

Reporting View 1 - Executive KPI Summary

In [0]:
%sql
CREATE OR REPLACE VIEW executive_kpi_summary AS
SELECT
   COUNT(DISTINCT facility_id) AS total_hospitals,
   COUNT(*) AS reporting_records,
   COUNT(DISTINCT measure_display_name) AS measures,
   ROUND(AVG(excess_readmission_ratio_numeric), 3) AS avg_err,
   ROUND(
      SUM(CASE
          WHEN reporting_status = 'Standard Reporting' THEN 1
          ELSE 0
      END) * 1 / COUNT(*),
      4
   ) AS pct_standard_reporting
FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL;

In [0]:
%sql
select * from executive_kpi_summary

total_hospitals,reporting_records,measures,avg_err,pct_standard_reporting
2833,11720,6,1.002,0.9678


Reporting View 2 - Measure KPI Summary

In [0]:
%sql
CREATE OR REPLACE VIEW measure_kpi_summary AS
SELECT
    measure_display_name,
    count(*) as records,
    ROUND(AVG(excess_readmission_ratio_numeric),3) AS avg_err,
    round(STDDEV(excess_readmission_ratio_numeric),3) AS err_variation,
    ROUND(MIN(excess_readmission_ratio_numeric),3) AS min_err,
    ROUND(MAX(excess_readmission_ratio_numeric),3) AS max_err
FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL
AND Number_of_Discharges_numeric >= 25
GROUP BY measure_display_name;

Reporting View 3 - State KPI Summary

In [0]:
%sql
CREATE OR REPLACE VIEW state_kpi_summary AS 
SELECT 
    state,
    count(*) AS reporting_records,
    ROUND(AVG(excess_readmission_ratio_numeric), 3) AS avg_err,
    ROUND(AVG(Predicted_Readmission_Rate_numeric), 2) AS avg_predicted,
    ROUND(AVG(Expected_Readmission_Rate_numeric), 2) AS avg_expected,
    CASE
    WHEN avg_err < 0.95 THEN 'Better Than Expected'
    WHEN avg_err <= 1.00 THEN 'Near Expected'
    ELSE 'Worse Than Expected'
END AS performance_tier

FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL
AND reporting_status = 'Standard Reporting'
GROUP BY state
ORDER BY State ASC


In [0]:
%sql
select * from measure_kpi_summary

measure_display_name,records,avg_err,err_variation,min_err,max_err
Coronary Artery Bypass Graft Surgery (CABG),363,1.029,0.113,0.743,1.424
Heart Attack,1240,1.01,0.072,0.684,1.264
COPD,1545,1.011,0.05,0.868,1.255
Pneumonia,2320,1.007,0.069,0.797,1.426
Heart Failure,2316,1.004,0.068,0.714,1.361
Hip/Knee Replacement,253,1.036,0.223,0.47,1.583


Reporting View 4 - Hospital Distribution / Benchmarking

In [0]:
%sql
CREATE OR REPLACE VIEW hospital_benchmarking AS
SELECT
    facility_name,
    state,
    measure_display_name,
    Number_of_Discharges_numeric,
    ROUND(excess_readmission_ratio_numeric,3) AS err,
        CASE
        WHEN excess_readmission_ratio_numeric >= 1.20 THEN '1.20+'
        WHEN excess_readmission_ratio_numeric >= 1.10 THEN '1.10-1.19'
        WHEN excess_readmission_ratio_numeric >= 1.00 THEN '1.00-1.09'
        ELSE 'Below 1.00'
    END AS err_band
FROM workspace.default.cms_readmissions_clean4
WHERE excess_readmission_ratio_numeric IS NOT NULL
AND reporting_status = 'Standard Reporting'
AND Number_of_Discharges_numeric >= 25
ORDER BY facility_name


### Reporting Quality Findings

Reporting completeness analysis showed that approximately 97% of records qualified as standard reporting records, supporting stable KPI benchmarking analysis. Suppressed and unavailable reporting categories represented a relatively small proportion of the dataset and were concentrated primarily within low-discharge reporting populations.

## Final Findings

National hospital readmission performance clustered closely around CMS expected benchmarks across analyzed measures. While most measures demonstrated relatively standardized hospital-level performance, Hip/Knee Replacement showed substantially greater variation across facilities. COPD readmission performance remained tightly clustered nationally, suggesting more operationally standardized management practices.

Reporting completeness exceeded 96%, supporting stable KPI benchmarking analysis after applying discharge-volume and reporting-quality filters. Final reporting views were prepared to support downstream Power BI visualization and healthcare KPI reporting workflows.